# Agent Evaluation

In [28]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new-v1.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [29]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [30]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [31]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_module_agentic_rag/.env")
from openai import OpenAI
import os
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [32]:
model = "qwen/qwen3.5-9b"

In [33]:
# Define search query tool
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [34]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model=model, client=openai_client)
)

In [35]:
runner

The result contains:

    last_message: the final response
    all_messages: the full message history
    cost: the cost of all LLM calls in this run


In [36]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/04_evaluation/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3.5-9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [37]:
result

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='I found the course last minute, is it too late to sign up?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"late sign up deadline register join after finding course"}', call_id='call_925814480550043', name='search', type='function_call', id='fc_8n3bb2uzfwmq26votk05m', caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_925814480550043', 'output': '[\n  {\n    "id": "cdc3b285e5",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Can I submit homework after the deadline, or get a deadline extension?",\n    "answer": "No. We don\'t give individual deadline extensions, and once the homework submission form i

In [38]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I found the course last minute, is it too late to sign up?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late sign up deadline register join after finding course"}', call_id='call_925814480550043', name='search', type='function_call', id='fc_8n3bb2uzfwmq26votk05m', caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_925814480550043',
  'output': '[\n  {\n    "id": "cdc3b285e5",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Can I submit homework after the deadline, or get a deadline extension?",\n    "answer": "No. We don\'t give individual deadline extensions, and once the homework submission form is closed you can 

For this lesson, the trajectory is only the tool calls. We don't need to send the full message history to the judge.

In [39]:
# Extracting function name and arguments
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [40]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"late sign up deadline register join after finding course"}'}]

In [41]:
# Getting original answer
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [42]:
import json

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": json.dumps(tool_calls),
    # "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I found the course last minute, is it too late to sign up?',
 'answer_agent': 'Yes, it\'s not too late! According to our FAQ:\n\n**"I just discovered the course. Can I still join?"**\n\n> Yes, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nSo you can join even if you found the course last minute. Just keep in mind that to get a certificate, you\'ll need to submit your final project before the submission deadline for that course closes. The videos and GitHub materials are available for you to start whenever you want.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': '[{"name": "search", "arguments": "{\\"query\\":\\"late sign up deadline register join after finding course\\"}"}]',
 'document': '74eb249bbf'}

In [43]:
## Processing multiple questions
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": json.dumps(tool_calls),
        # "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [44]:
# Test on a few questions
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=16) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/04_evaluation/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3.5-9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/04_evaluation/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3.5-9b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
/home/user1129/MyDocuments/10_14_Life_Admin/13_Technology/100_Coding/100_01_Python/llm_ai_engineering/llm_zoomcamp_portfolio/modules/04_evaluation/.venv/lib/python3.14/site-packages/toyaikit

In [47]:
df_agent = pd.DataFrame(agent_answers)

In [ ]:
df_agent["cost"].sum() #localai model, no cost

In [48]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [49]:
# load the agent answer data
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

Our judge can look at both:

    whether answer_agent matches answer_orig
    whether the tool calls look reasonable for the question

This lets us evaluate the final answer and the agent behavior in one place.

For our search agent, a good trajectory has these properties:

    The search query is relevant to the user question
    The query includes the important keywords from the question
    The agent avoids duplicate searches with the same arguments
    If it searches more than once, the next query is a useful refinement
    It usually uses 1 search call
    2-3 calls can be okay for harder questions
    More than 3 search calls needs a clear reason
    The tool calls support the final answer
    The agent does not stop too early or keep searching without a reason


In [50]:
# Defining a judge for the agent

from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [51]:
# Instructions
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [55]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry_chat_completions

def evaluate_agent_answer(rec, model=model):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry_chat_completions(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [56]:
# testing
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning="The agent answer contradicts the original answer. The FAQ clearly states 'Yes, but...' implying it is possible to sign up but with conditions for certificates. The agent claims 'No, it's not too late' and misinterprets the FAQ content about 'starting without registering'. This shows a fundamental mismatch in meaning.", answer_score='bad', trajectory_reasoning="The search query contains relevant keywords ('too late sign up course start') and seems to have retrieved the correct source of information (the FAQ). However, the agent failed to correctly interpret the retrieved text.", trajectory_score='good')


When the answer is bad, the trajectory score tells us whether the problem started with tool use. If the answer is bad but the trajectory is good, the model may have used the retrieved context poorly. If both are bad, the agent likely searched for the wrong thing. It may also have stopped too early.

In [57]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [58]:
# Testing on first 50 questions
with ThreadPoolExecutor(max_workers=16) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [59]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [60]:
agent_evaluations

[{'question': 'I found the course last minute, is it too late to sign up?',
  'document': '74eb249bbf',
  'answer_score': 'bad',
  'answer_reasoning': "The original answer states that signing up (receiving a certificate) is possible if the project submission deadline hasn't passed. It does not explicitly say it is 'not too late' in a general sense, but implies a conditional 'yes'. The agent's answer contradicts this by stating 'No, it's not too late', which is misleading because if the course has closed for registration or project submissions, one *would* be too late. The key constraint mentioned in the ground truth ('while we're still accepting submissions') is missing from the agent's response. Therefore, the semantic meaning is incorrect despite the tool call.",
  'trajectory_score': 'good',
  'trajectory_reasoning': "The agent made only one search call with a reasonable query related to the user's question. While there was only one call, the result of that call (which we must assum

In [61]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [62]:
df_agent_eval

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,"I found the course last minute, is it too late...",74eb249bbf,bad,The original answer states that signing up (re...,good,The agent made only one search call with a rea...
1,"Can I still join if I start now, but what abou...",74eb249bbf,good,The agent's answer aligns well with the origin...,good,The agent made only one search call with a con...
2,Is it possible to enroll even though the deadl...,74eb249bbf,bad,"The original answer explicitly states: 'Yes, b...",good,The agent made only one tool call with the que...
3,"Hey, I just saw this—am I eligible to join and...",74eb249bbf,bad,The agent's answer contains specific details (...,good,The agent made only one search call. This is a...
4,"Since I discovered the course recently, can I ...",74eb249bbf,good,The agent's answer accurately reflects the gro...,good,The agent made a single search call with a rel...
5,"Hey, I signed up for LLM Zoomcamp but didn't g...",977bf7786c,good,The agent's answer directly addresses the user...,good,The agent made exactly one search call with a ...
6,Is there really a need to register if I just w...,977bf7786c,bad,The agent's answer directly contradicts the or...,bad,The agent made only one search tool call. Whil...
7,Does anyone actually check against a registere...,977bf7786c,good,The agent's final answer directly contradicts ...,good,The agent made exactly one search call. This i...
8,"So basically, registration is just for them to...",977bf7786c,good,The agent's answer aligns perfectly with the o...,good,The agent made a single search call with a spe...
9,Can I just start learning right away without w...,977bf7786c,good,The agent answer is accurate and aligns well w...,good,


In [ ]:
calc_total_price(usages)

In [63]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    44
bad      6
Name: count, dtype: int64

In [64]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    48
bad      2
Name: count, dtype: int64

In [65]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)